# Find Missing Quarters per Store

Given a table `b_sql.b_practice.STORES` with columns `Store`, `Quarter` (`'Q1'`–`'Q4'`), and `Amount`, find the **missing quarter(s)** for each store — i.e., quarters where the store has no sales record.

Solved multiple ways:
1. **SQL Arithmetic** — sum the quarter numbers; subtract from 10 to derive the missing quarter.
2. **SQL Recursive CTE** — generate all store-quarter pairs, then `EXCEPT` existing ones.
3. **PySpark** — equivalent logic using `sequence`/`explode` + `exceptAll`, and the arithmetic approach.

In [0]:
%sql
-- Create and populate the STORES table
CREATE TABLE b_sql.b_practice.STORES (
Store varchar(10),
Quarter varchar(10),
Amount int);

INSERT INTO b_sql.b_practice.STORES (Store, Quarter, Amount)
VALUES ('S1', 'Q1', 200),
('S1', 'Q2', 300),
('S1', 'Q4', 400),
('S2', 'Q1', 500),
('S2', 'Q3', 600),
('S2', 'Q4', 700),
('S3', 'Q1', 800),
('S3', 'Q2', 750),
('S3', 'Q3', 900);

In [0]:
%sql
-- Preview the STORES table
select * from b_sql.b_practice.stores

In [0]:
%sql
-- SQL Method 1: Find missing quarter using arithmetic (10 - sum of existing quarter numbers)
WITH CTE AS(
SELECT *,(CAST(SUBSTR(Quarter,2,1) As Int)) AS Existing_Quarter
FROM b_sql.b_practice.STORES)
SELECT Store,CONCAT('Q',(10-SUM(Existing_Quarter))) AS missing_Quarter
FROM CTE
GROUP BY Store

In [0]:
%sql
-- SQL Method 2: Find missing quarter using recursive CTE + EXCEPT
with recursive cte as(
select distinct store,1 as q from b_sql.b_practice.STORES
union all
select store,q+1 as q from cte where cte.q<4)
,cte1 as(
select store,q from cte 
except 
select store,right(quarter,1) from b_sql.b_practice.STORES)
select store,concat('Q',cast(q as string)) as quarter from cte1



In [0]:
# Load the STORES table into a PySpark DataFrame
df=spark.read.table("b_sql.b_practice.STORES")
df.display()

In [0]:
# Import PySpark functions and types
from pyspark.sql.functions import *
from pyspark.sql.types import *



In [0]:
# Generate all store-quarter pairs (Q1-Q4) using sequence + explode
df_r=df.select("Store").distinct().withColumn("quarter",sequence(lit(1),lit(4))).withColumn("quarter",explode("quarter"))
df_r.display()


In [0]:
# PySpark Method 1: Find missing quarters using exceptAll (all pairs minus existing)
df_r.select("Store",concat(lit("Q"),col("quarter")).alias("Quarter")).exceptAll(df.select("Store",col("quarter"))).display()

In [0]:
# PySpark Method 2: Find missing quarter using arithmetic (10 - sum of existing quarter numbers)
from pyspark.sql.functions import col, substring, sum, concat, lit

df_result = (
    df
    .withColumn(
        "Existing_Quarter",
        substring(col("Quarter"), 2, 1).cast("int")
    )
    .groupBy("Store")
    .agg(
        sum("Existing_Quarter").alias("quarter_sum")
    )
    .withColumn(
        "missing_Quarter",
        concat(lit("Q"), (lit(10) - col("quarter_sum")))
    )
    .select("Store", "missing_Quarter")
)
df_result.display()
